# 5. Evaluation and Inference (Standard VQ-VAE)

This notebook serves as the final validation stage for the baseline version of our project, utilizing a standard Vector Quantized VAE (VQ-VAE).
Our objective is to quantify the performance of the LLM when it is forced to reason through a traditional Euclidean discrete space. Unlike the "Latent Oddity" approach, the quantization here is learned end-to-end with the VAE, providing a direct comparison between a standard discrete bottleneck and one informed by stochastic Riemannian geometry.

## 5.1 Environment Setup and Repository Cloning

To ensure reproducibility, this section automates the setup of the working environment:
1. **Google Drive Integration:** Mounts your personal Drive to store persistent data (checkpoints and processed datasets).
2. **Project Structure:** Automatically creates a `DLAI` folder in your Drive.
3. **Dependency Management:** Installs the `uv` package manager and resolves all requirements defined in `pyproject.toml`.
4. **Source Code:** Clones the `llama` branch from our GitHub repository to provide access to the `src` module and configuration files.

**Note for Evaluators:** Please authorize the Google Drive mount when prompted to allow the notebook to save and retrieve project files.

In [ ]:
import os, sys

# 1. Mount Google Drive
# Evaluators will need to accept the pop-up to connect their Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Setup directories on Drive
# Create the DLAI folder if it doesn't exist on their Drive
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/DLAI"
if not os.path.exists(DRIVE_PROJECT_PATH):
    os.makedirs(DRIVE_PROJECT_PATH, exist_ok=True)
    print(f"Created project folder at: {DRIVE_PROJECT_PATH}")

# 3. UV Installation
# We use UV for much faster dependency management than standard pip
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ['PATH'] = f"{os.path.expanduser('~')}/.cargo/bin:" + os.environ['PATH']

# 4. Clone the Repository (Branch: llama)
# If the local folder doesn't exist, clone the specific branch
%cd /content
if not os.path.exists("DLAI"):
    !git clone --branch llama https://github.com/irene-30/DLAI.git
else:
    print("Repo already exists, pulling latest changes...")
    !git -C DLAI pull

# 5. Synchronize pyproject.toml
# Copy the pyproject.toml from the cloned repo to the Drive folder (if necessary)
# or vice versa, to ensure that UV reads the correct dependencies.
!cp /content/DLAI/pyproject.toml {DRIVE_PROJECT_PATH}/pyproject.toml

# 6. Install dependencies via pyproject.toml
# This command reads the .toml file and installs everything necessary
%cd /content/DLAI
!uv pip install -e . --system

# 7. Add to the system path to allow imports from 'src'
sys.path.append("/content/DLAI")
%cd /content

print("✅ Setup completed successfully!")

Mounted at /content/drive
downloading uv 0.11.13 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
/content
Cloning into 'DLAI'...
remote: Enumerating objects: 637, done.
remote: Counting objects: 100% (191/191), done.
remote: Compressing objects: 100% (191/191), done.
remote: Total 637 (delta 135), reused 0 (delta 0), pack-reused 446 (from 2)
Receiving objects: 100% (637/637), 280.11 KiB | 11.20 MiB/s, done.
Resolving deltas: 100% (370/370), done.
/content/DLAI
Using Python 3.12.13 environment at: /usr
Resolved 88 packages in 1.38s
Prepared 6 packages in 6.29s
Uninstalled 3 packages in 217ms
Installed 6 packages in 108ms
 + bitsandbytes==0.49.2
 - datasets==4.0.0
 + datasets==4.8.5
 + dlai-metamath==0.1.0 (from file:///content/DLAI)
 - pyarrow==18.1.0
 + pyarrow==24.0.0
 - torchao==0.10.0
 + torchao==0.17.0
 + trl==1.4.0
/content


In [ ]:
# --- 1. Global Imports and Setup ---
import torch
import json
import re
import shutil
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM
from peft import PeftModel

from src.utils import get_llm_tokenizer, MAX_SEQ_LEN, VQ_CODEBOOK_SIZE, parse_gsm8k_sample, extract_final_answer
from src.model.vae import VQVAEModel

# --- PATHS ---
LLM_MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
PATH_LLM_MODEL_ODDITY = "/content/drive/MyDrive/DLAI/experiments/vqvae_standard/llama_finetuning_results/final_model"
PATH_CONTINUOUS_VAE = "/content/drive/MyDrive/DLAI/experiments/vqvae_standard/vqvae_final.pth"
SAVE_DIR = "/content/drive/MyDrive/DLAI/experiments/vqvae_standard/evaluate/"

os.makedirs(SAVE_DIR, exist_ok=True)
print(f"✅ Directory ready: {SAVE_DIR}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")

# Load Tokenizer
llm_tokenizer = get_llm_tokenizer()

/usr/local/lib/python3.12/dist-packages/torchao/quantization/quant_api.py:1745: SyntaxWarning: invalid escape sequence '\.'
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.


✅ Directory ready: /content/drive/MyDrive/DLAI/experiments/vqvae_standard/evaluate/
✅ Using device: cuda
Loading tokenizer: meta-llama/Llama-3.2-3B-Instruct


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

### Step 1: Quantitative Evaluation (GSM8K)
In this section, we measure the model's accuracy on the GSM8K benchmark to establish a performance baseline. The evaluation loop prompts the fine-tuned LLM and isolates the final numerical answer from the generated response. To ensure robustness against potential environment timeouts, we implement a persistent checkpointing system that allows us to resume the evaluation from the last processed sample.

In [ ]:
# --- 2. Evaluation Logic ---
def evaluate_model(model_path):
    primary_file = os.path.join(SAVE_DIR, "eval_results.json")
    backup_file = os.path.join(SAVE_DIR, "eval_results_backup.json")

    print(f"--- 📊 Evaluating Model from {model_path} ---")

    # Load Checkpoint / Progress
    correct = 0
    total = 0
    start_idx = 0

    load_path = None
    if os.path.exists(primary_file):
        load_path = primary_file
    elif os.path.exists(backup_file):
        load_path = backup_file
        print("⚠️ Primary checkpoint missing. Loading from Backup.")

    if load_path:
        try:
            with open(load_path, "r") as f:
                ckpt = json.load(f)
                correct = ckpt.get("correct", 0)
                total = ckpt.get("total", 0)
                start_idx = total
                print(f"🔄 Resuming from sample {start_idx}...")
        except Exception as e:
            print(f"❌ Could not load checkpoint ({e}). Starting fresh.")

    # Load Model & Tokenizer
    print(f"Loading Base Model: {LLM_MODEL_NAME}")
    base_model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL_NAME,
        torch_dtype="auto",
        device_map="auto"
    )

    # Resize before applying LoRA
    base_model.resize_token_embeddings(len(llm_tokenizer))

    print(f"Loading LoRA adapters from: {model_path}")
    eval_model = PeftModel.from_pretrained(base_model, model_path)
    eval_model.eval()

    # Load & Slice Test Data
    full_test_data = load_dataset("gsm8k", "main")['test']
    if start_idx >= len(full_test_data):
        print("✅ Evaluation already complete.")
        return

    test_data = full_test_data.select(range(start_idx, len(full_test_data)))
    print(f"Loaded {len(full_test_data)} test samples. Remaining to process: {len(test_data)}.")

    pbar = tqdm(test_data, desc="Evaluating", initial=start_idx, total=len(full_test_data))

    for i, sample in enumerate(pbar):
        current_global_idx = start_idx + i

        parsed = parse_gsm8k_sample(sample)
        if not parsed:
            pbar.write(f"Skipping sample {current_global_idx}: Failed to parse.")
            total += 1
            continue

        prompt = parsed['prompt']
        solution = parsed['solution']
        true_answer = extract_final_answer(solution)

        if true_answer is None:
            pbar.write(f"Skipping sample {current_global_idx}: No true answer found.")
            total += 1
            continue

        # Generate response
        inputs = llm_tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            output = eval_model.generate(
                **inputs,
                max_new_tokens=256,
                pad_token_id=llm_tokenizer.pad_token_id,
                eos_token_id=llm_tokenizer.eos_token_id
            )

        generated_text = llm_tokenizer.decode(output[0], skip_special_tokens=True)
        pred_answer = extract_final_answer(generated_text)

        # Check correctness
        if pred_answer is not None and pred_answer == true_answer:
            correct += 1
        total += 1

        current_acc = (correct / total) * 100 if total > 0 else 0
        pbar.set_postfix({"acc": f"{current_acc:.2f}%", "correct": correct, "total": total})

        # Rolling Checkpoint (Save every 50 samples)
        if total % 50 == 0:
            if os.path.exists(primary_file):
                shutil.copy2(primary_file, backup_file)

            results = {
                "accuracy": current_acc,
                "correct": correct,
                "total": total,
                "status": "in_progress"
            }
            with open(primary_file, "w") as f:
                json.dump(results, f, indent=4)

    # Final Report & Save
    accuracy = (correct / total) * 100 if total > 0 else 0
    print("\n" + "="*30)
    print("📈 FINAL EVALUATION RESULTS")
    print(f"Correct: {correct} | Total: {total}")
    print(f"Accuracy: {accuracy:.2f}%")
    print("="*30)

    final_results = {
        "accuracy": accuracy,
        "correct": correct,
        "total": total,
        "status": "completed"
    }

    if os.path.exists(primary_file):
        shutil.copy2(primary_file, backup_file)

    with open(primary_file, "w") as f:
        json.dump(final_results, f, indent=4)
    print(f"✅ Final results saved to {primary_file}.")

    # Free up memory before inference
    del eval_model
    del base_model
    torch.cuda.empty_cache()

# Execute Evaluation
evaluate_model(model_path=PATH_LLM_MODEL_ODDITY)

--- 📊 Evaluating Model from /content/drive/MyDrive/DLAI/experiments/vqvae_standard/llama_finetuning_results/final_model ---
Loading Base Model: meta-llama/Llama-3.2-3B-Instruct


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Loaded 1319 test samples. Remaining to process: 1319.


Evaluating:   0%|          | 0/1319 [00:00<?, ?it/s]


📈 FINAL EVALUATION RESULTS
Correct: 541 | Total: 1319
Accuracy: 41.02%
✅ Final results saved to /content/drive/MyDrive/DLAI/experiments/vqvae_standard/evaluate/eval_results.json.


### Step 2: Qualitative Inference (Standard Generation)
We load the LoRA adapters onto the base Llama-3 model to observe its generation behavior in real-time. By providing a mathematical reasoning prompt, we can verify if the LLM has successfully learned to incorporate the `<latent_N>` tokens into its response. This qualitative check is essential to confirm that the model is actually utilizing the standard VQ-VAE codebook as an intermediate "thinking" step before concluding the problem.

In [ ]:
# --- 3. Generate a response containing Riemannian tokens ---
print(f"--- 🛠️ Initializing Riemannian Model for Inference ---")

# Load the BASE model first
base_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,
    torch_dtype=torch.float16, # Use float16 to fit on GPU safely
    device_map="auto"
)

# Resize BEFORE loading the adapter to accommodate custom Riemannian tokens
base_model.resize_token_embeddings(len(llm_tokenizer))

# Load your custom "Latent Oddity" adapter onto the resized base
llm_model = PeftModel.from_pretrained(base_model, PATH_LLM_MODEL_ODDITY)
llm_model.eval()

print("✅ Riemannian Model loaded and resized successfully.")

question = "Mark has $50. He buys 3 books that cost $7 each. How much money does he have left?"
prompt = f"Question: {question}\nAnswer: "

print("\n--- 🧠 Generating Riemannian Response ---")
inputs = llm_tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    output = llm_model.generate(
        **inputs,
        max_new_tokens=150,
        pad_token_id=llm_tokenizer.pad_token_id,
        eos_token_id=llm_tokenizer.eos_token_id,
        temperature=0.7,
        do_sample=True
    )

# skip_special_tokens=False allows us to see the latent 'oddity' tokens in the string
generated_text = llm_tokenizer.decode(output[0], skip_special_tokens=False)

print("\n--- GENERATED RESPONSE (RIEMANNIAN TOKENS) ---")
print(generated_text)
print("-" * 50)

--- 🛠️ Initializing Riemannian Model for Inference ---


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


✅ Riemannian Model loaded and resized successfully.

--- 🧠 Generating Riemannian Response ---

--- GENERATED RESPONSE (RIEMANNIAN TOKENS) ---
<|begin_of_text|>Question: Mark has $50. He buys 3 books that cost $7 each. How much money does he have left?
Answer: 30 - 21 = #### 9
Mark has $9 left. #### 9<|eot_id|>
--------------------------------------------------


### Step 3: Decoding the Latent Space
This final step acts as a "window" into the model's discrete reasoning. We extract the generated `<latent_N>` IDs and map them back to their embeddings using the VQ-VAE's internal codebook. Because the standard VQ-VAE integrates the quantization and decoding layers into a single architecture, the interpretation process is more direct than in the "Oddity" pipeline, allowing us to immediately see how the continuous decoder translates these Euclidean points back into natural language.

In [ ]:
# --- 4. Extract and Decode Standard VQ-VAE Thoughts ---
import torch
import re

print("--- 🔬 Initializing Standard VQ-VAE Backbone ---")

# Initialize VQVAE
vae_model = VQVAEModel(
    vocab_size=len(llm_tokenizer),
    d_model=256,
    max_seq_len=MAX_SEQ_LEN
).to(device)

# Load weights
vae_model.load_state_dict(torch.load(PATH_CONTINUOUS_VAE, map_location=device))
vae_model.eval()

# Extract tokens generated by LLM
latent_token_ids = [int(i) for i in re.findall(r"<latent_(\d+)>", generated_text)]

if latent_token_ids:
    print(f"🔍 Found {len(latent_token_ids)} standard latent tokens. Interpreting...")
    indices_tensor = torch.tensor(latent_token_ids, dtype=torch.long).to(device)

    with torch.no_grad():
        # In the standard VQ-VAE, the codebook is inside the model (es. vae_model.quantize.embedding)
        codebook_embeddings = vae_model.quantize.embedding(indices_tensor)
        quantized_memory = codebook_embeddings.unsqueeze(0)

        # Direct decode
        logits = vae_model.decode(quantized_memory)
        predicted_token_ids = torch.argmax(logits, dim=-1).squeeze(0)

        interpreted_text = llm_tokenizer.decode(predicted_token_ids, skip_special_tokens=True)

    print("\n--- INTERPRETATION OF STANDARD VQ-VAE THOUGHTS ---")
    print(f"Raw Latent Sequence: {latent_token_ids}")
    print(f"Decoded Interpretation:\n{interpreted_text}")
    print("-" * 50)
else:
    print("\nℹ️ No latent tokens found.")

--- 🔬 Initializing Standard VQ-VAE Backbone ---

ℹ️ No latent tokens found.
